# Evaluation and explainability workbench

This Kaggle-ready notebook evaluates agent scores, compares the meta-learner, and generates auditable explanations. It uses a small fallback fixture when generated Parquet data is not mounted.

In [ ]:
import logging
import random
import numpy as np
import torch

# GPU/TPU detection with CPU fallback
device = "cuda" if torch.cuda.is_available() else "cpu"
logging.info("Using device: %s", device)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Structured logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logging.info("Phase 5 evaluation and explainability workbench started")

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path('/kaggle/working/neural-sentinel')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.agents.explanation_agent import ExplanationAgent
from src.evaluation.agent_evaluation import AgentEvaluator

In [ ]:
data_path = ROOT / 'data' / 'generated' / 'transactions_5m.parquet'
if data_path.exists():
    transactions = pd.read_parquet(data_path)
else:
    transactions = pd.DataFrame({
        'transaction_id': [f'T{i}' for i in range(10)],
        'transaction_type': ['transfer'] * 10,
        'amount_npr': [950000, 100, 1200000, 250, 980000, 80, 700000, 300, 1500000, 200],
        'channel': ['mobile_banking'] * 10,
        'remittance_corridor': ['Qatar->Nepal', 'domestic', 'India->Nepal', 'domestic', 'Qatar->Nepal', 'domestic', 'domestic', 'domestic', 'India->Nepal', 'domestic'],
        'is_fraud': [1, 0, 1, 0, 1, 0, 0, 0, 1, 0],
        'velocity_score': [0.90, 0.05, 0.40, 0.02, 0.80, 0.01, 0.20, 0.03, 0.50, 0.01],
        'kyc_aml_score': [0.95, 0.01, 0.50, 0.01, 0.90, 0.01, 0.10, 0.01, 0.60, 0.01],
        'meta_risk_score': [0.92, 0.03, 0.68, 0.02, 0.88, 0.01, 0.12, 0.02, 0.75, 0.01],
        'kyc_aml_reason_code': ['STRUCTURING', 'NO_VIOLATIONS', 'CROSS_BORDER', 'NO_VIOLATIONS', 'STRUCTURING', 'NO_VIOLATIONS', 'NO_VIOLATIONS', 'NO_VIOLATIONS', 'CROSS_BORDER', 'NO_VIOLATIONS'],
    })

score_columns = {
    name.removesuffix('_score'): name
    for name in transactions.columns if name.endswith('_score')
}
metrics = AgentEvaluator(precision_at_k_fraction=0.01).evaluate(transactions, score_columns)
metrics

In [ ]:
evaluator = AgentEvaluator()
# The display name is derived from meta_risk_score as meta_risk.
system_summary = evaluator.compare_system(metrics, meta_name='meta_risk')
system_summary

In [ ]:
explanations = ExplanationAgent().fit(transactions).predict(transactions)
explanations[['transaction_id', 'risk_score', 'alert_flag', 'explanation']].head()

## Kaggle-ready checklist

- Keep the notebook self-contained with a small fallback fixture when generated Parquet data is unavailable.
- Import production logic from `src/` instead of duplicating it here.
- Use the same canonical columns as the agent contract so notebook results match automated tests.
- Prefer concise, auditable explanations and review them against the suspicious-activity framing in `AGENTS.md`.